# Apache Hudi - Deletes & Soft Deletes

Apache Hudi supports record-level deletes, which makes it possible to remove data from a data lake without rewriting an entire dataset manually.

In real-world systems, deletes are usually implemented in two different ways: hard deletes physically remove records from the latest table snapshot, while soft deletes keep the record but mark it as deleted using a business flag.


## What you will learn

In this notebook, you will learn:

- Difference between hard delete and soft delete patterns
- Creating an independent Hudi Copy-On-Write (COW) table for delete examples
- Performing a hard delete using `hoodie.datasource.write.operation = delete`
- Using record key, partition path, and precombine field correctly during deletes
- Implementing a soft delete with an `is_deleted` column
- Querying only active records after soft deletes
- Inspecting Hudi metadata columns after delete operations
- Understanding delete/tombstone semantics at a practical level

## Step 1 — Create Spark session

This notebook assumes that the Hudi bundle JAR is already available in the Docker image under Spark's `jars` directory.

Because of that, we do **not** use `spark.jars.packages` here. This avoids Ivy/Maven downloads every time the notebook starts.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-03-Deletes-Soft-Deletes")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Spark version: 4.0.2
Spark master: spark://spark-master:7077


## Step 2 — Define table configuration

Each notebook in this Hudi series is self-contained. Notebook 3 creates and uses its own table, so it does not depend on Notebook 1 or Notebook 2.

In [2]:
BASE_PATH = "/workspace/data/hudi"
TABLE_NAME = "riders_cow_deletes"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)

Table name: riders_cow_deletes
Table path: /workspace/data/hudi/riders_cow_deletes


## Step 3 — Clean previous local run

This step makes the notebook repeatable. It removes only the table folder used by this notebook.

In [3]:
import shutil
from pathlib import Path

path = Path(TABLE_PATH)

if path.exists():
    shutil.rmtree(path)
    print(f"Removed existing table path: {TABLE_PATH}")
else:
    print("No previous table path found. Starting clean.")

No previous table path found. Starting clean.


## Step 4 — Prepare baseline data

We start with three riders. The `is_deleted` column is included from the beginning so that soft delete updates can use the same table schema later.

In [4]:
from pyspark.sql.functions import to_timestamp

baseline_data = [
    ("r1", "SFO", "2024-01-01 10:00:00", False),
    ("r2", "NYC", "2024-01-01 11:00:00", False),
    ("r3", "LON", "2024-01-01 12:00:00", False),
]

baseline_df = spark.createDataFrame(
    baseline_data,
    ["rider", "city", "ts", "is_deleted"]
).withColumn("ts", to_timestamp("ts"))

baseline_df.show(truncate=False)
baseline_df.printSchema()

[Stage 1:===================>                                       (1 + 2) / 3]

+-----+----+-------------------+----------+
|rider|city|ts                 |is_deleted|
+-----+----+-------------------+----------+
|r1   |SFO |2024-01-01 10:00:00|false     |
|r2   |NYC |2024-01-01 11:00:00|false     |
|r3   |LON |2024-01-01 12:00:00|false     |
+-----+----+-------------------+----------+

root
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- is_deleted: boolean (nullable = true)



## Step 5 — Write the baseline Hudi COW table

The table uses:

- `rider` as the record key
- `city` as the partition path
- `ts` as the precombine field

The precombine field helps Hudi decide which version of a record should win when multiple rows have the same record key.

In [5]:
hudi_common_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.table.type": "COPY_ON_WRITE",
}

(
    baseline_df.write
    .format("hudi")
    .options(**hudi_common_options)
    .mode("overwrite")
    .save(TABLE_PATH)
)

print("Baseline Hudi table created.")

# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope


[Stage 35:>                                                         (0 + 3) / 3]

26/04/25 19:33:31 WARN HoodieTableFileSystemView: Partition: NYC is not available in store
26/04/25 19:33:31 WARN HoodieTableFileSystemView: Partition: LON is not available in store
26/04/25 19:33:31 WARN HoodieTableFileSystemView: Partition: SFO is not available in store


Baseline Hudi table created.


## Step 6 — Read and verify the baseline table

Hudi adds internal metadata columns prefixed with `_hoodie_`. These columns are useful for understanding commits, record keys, partitions, and physical files.

In [6]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_deletes")

spark.sql("""
SELECT
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path,
    rider,
    city,
    ts,
    is_deleted
FROM riders_deletes
ORDER BY rider
""").show(truncate=False)

+-------------------+------------------+----------------------+-----+----+-------------------+----------+
|_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|rider|city|ts                 |is_deleted|
+-------------------+------------------+----------------------+-----+----+-------------------+----------+
|20260425193312506  |r1                |SFO                   |r1   |SFO |2024-01-01 10:00:00|false     |
|20260425193312506  |r2                |NYC                   |r2   |NYC |2024-01-01 11:00:00|false     |
|20260425193312506  |r3                |LON                   |r3   |LON |2024-01-01 12:00:00|false     |
+-------------------+------------------+----------------------+-----+----+-------------------+----------+



## Step 7 — Capture commits after the baseline insert

Instead of reading timeline files directly from `.hoodie`, this notebook reads commit information from Hudi metadata columns. This is more robust across Hudi versions.

In [7]:
def list_commit_times(table_path):
    df = spark.read.format("hudi").load(table_path)
    return [
        row["_hoodie_commit_time"]
        for row in (
            df.select("_hoodie_commit_time")
            .distinct()
            .orderBy("_hoodie_commit_time")
            .collect()
        )
    ]

baseline_commits = list_commit_times(TABLE_PATH)

print("Commits after baseline insert:")
for commit in baseline_commits:
    print(commit)

Commits after baseline insert:
20260425193312506


## Step 8 — Perform a hard delete

A hard delete removes the record from the latest snapshot of the table.

For a partitioned table, include the partition field (`city`) in the delete DataFrame so Hudi can locate the record correctly.

In [8]:
hard_delete_df = spark.createDataFrame(
    [("r2", "NYC", "2024-01-03 09:00:00")],
    ["rider", "city", "ts"]
).withColumn("ts", to_timestamp("ts"))

hard_delete_df.show(truncate=False)

(
    hard_delete_df.write
    .format("hudi")
    .options(**hudi_common_options)
    .option("hoodie.datasource.write.operation", "delete")
    .mode("append")
    .save(TABLE_PATH)
)

print("Hard delete completed for rider r2.")

+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r2   |NYC |2024-01-03 09:00:00|
+-----+----+-------------------+



Hard delete completed for rider r2.


## Step 9 — Verify the hard delete

The latest snapshot should no longer contain `r2`.

In [9]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_deletes")

spark.sql("""
SELECT rider, city, ts, is_deleted, _hoodie_commit_time
FROM riders_deletes
ORDER BY rider
""").show(truncate=False)

spark.sql("""
SELECT rider, city, ts, is_deleted
FROM riders_deletes
WHERE rider = 'r2'
""").show(truncate=False)

+-----+----+-------------------+----------+-------------------+
|rider|city|ts                 |is_deleted|_hoodie_commit_time|
+-----+----+-------------------+----------+-------------------+
|r1   |SFO |2024-01-01 10:00:00|false     |20260425193312506  |
|r3   |LON |2024-01-01 12:00:00|false     |20260425193312506  |
+-----+----+-------------------+----------+-------------------+

26/04/25 19:33:57 WARN CacheManager: Asked to cache already cached data.
+-----+----+---+----------+
|rider|city|ts |is_deleted|
+-----+----+---+----------+
+-----+----+---+----------+



## Step 10 — Perform a soft delete

A soft delete does not remove the record physically from the latest snapshot. Instead, it updates the record and marks it as deleted using a business column such as `is_deleted = true`.

This pattern is useful when downstream systems need auditability, recoverability, or explicit delete markers.

In [10]:
soft_delete_df = spark.createDataFrame(
    [("r1", "SFO", "2024-01-04 10:00:00", True)],
    ["rider", "city", "ts", "is_deleted"]
).withColumn("ts", to_timestamp("ts"))

soft_delete_df.show(truncate=False)

(
    soft_delete_df.write
    .format("hudi")
    .options(**hudi_common_options)
    .option("hoodie.datasource.write.operation", "upsert")
    .mode("append")
    .save(TABLE_PATH)
)

print("Soft delete completed for rider r1.")

+-----+----+-------------------+----------+
|rider|city|ts                 |is_deleted|
+-----+----+-------------------+----------+
|r1   |SFO |2024-01-04 10:00:00|true      |
+-----+----+-------------------+----------+



Soft delete completed for rider r1.


## Step 11 — Verify the soft delete

The record still exists, but it is marked as deleted.

In [11]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_deletes")

spark.sql("""
SELECT rider, city, ts, is_deleted, _hoodie_commit_time
FROM riders_deletes
WHERE rider = 'r1'
""").show(truncate=False)

26/04/25 19:34:17 WARN CacheManager: Asked to cache already cached data.
+-----+----+-------------------+----------+-------------------+
|rider|city|ts                 |is_deleted|_hoodie_commit_time|
+-----+----+-------------------+----------+-------------------+
|r1   |SFO |2024-01-04 10:00:00|true      |20260425193359997  |
+-----+----+-------------------+----------+-------------------+



## Step 12 — Query only active records

Applications usually filter out soft-deleted rows at read time.

In [12]:
spark.sql("""
SELECT rider, city, ts, is_deleted
FROM riders_deletes
WHERE is_deleted = false
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+----------+
|rider|city|ts                 |is_deleted|
+-----+----+-------------------+----------+
|r3   |LON |2024-01-01 12:00:00|false     |
+-----+----+-------------------+----------+



## Step 13 — Inspect commits after delete operations

The table now has multiple commit instants: one for the baseline insert, one for the hard delete, and one for the soft delete upsert.

In [13]:
all_commits = list_commit_times(TABLE_PATH)

print("All commits visible in the latest table snapshot:")
for commit in all_commits:
    print(commit)

All commits visible in the latest table snapshot:
20260425193312506
20260425193359997


## Step 14 — Inspect Hudi metadata columns and physical files

The `_hoodie_file_name` column shows which physical file contains each visible record in the latest snapshot.

With Copy-On-Write tables, Hudi rewrites affected Parquet files during updates and deletes. The latest snapshot hides hard-deleted rows.

In [14]:
spark.sql("""
SELECT
    rider,
    city,
    is_deleted,
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path,
    _hoodie_file_name
FROM riders_deletes
ORDER BY rider
""").show(truncate=False)

+-----+----+----------+-------------------+------------------+----------------------+--------------------------------------------------------------------------+
|rider|city|is_deleted|_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|_hoodie_file_name                                                         |
+-----+----+----------+-------------------+------------------+----------------------+--------------------------------------------------------------------------+
|r1   |SFO |true      |20260425193359997  |r1                |SFO                   |84595e80-b7d7-4b40-bf35-8057c98039e1-0_0-110-578_20260425193359997.parquet|
|r3   |LON |false     |20260425193312506  |r3                |LON                   |326c6db0-785a-4bf1-b606-a5f573ab79e4-0_1-33-173_20260425193312506.parquet |
+-----+----+----------+-------------------+------------------+----------------------+--------------------------------------------------------------------------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 32882)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin

## Step 15 — Summary

In this notebook, you created a self-contained Hudi COW table and tested two delete strategies:

- **Hard delete**: the record is removed from the latest snapshot.
- **Soft delete**: the record remains but is marked with `is_deleted = true`.

Hard deletes are useful when data must disappear from query results. Soft deletes are useful when you need auditability or downstream systems need to understand delete events explicitly.